# VacationPy
---

## Starter Code to Import Libraries and Load the Weather and Coordinates Data

In [9]:
pip install geoviews

   ---------------------------------------- 0.0/547.3 kB ? eta -:--:--
   ---------------------------------------- 547.3/547.3 kB 6.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ------ --------------------------------- 1.8/11.0 MB 9.1 MB/s eta 0:00:01
   -------------- ------------------------- 3.9/11.0 MB 9.4 MB/s eta 0:00:01
   --------------------- ------------------ 5.8/11.0 MB 9.3 MB/s eta 0:00:01
   ---------------------------- ----------- 7.9/11.0 MB 9.4 MB/s eta 0:00:01
   ------------------------------------ --- 10.0/11.0 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 8.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ----------- ---------------------------- 1.8/6.3 MB 9.1 MB/s eta 0:00:01
   ----------------------- ---------------- 3.7/6.3 MB 9.1 MB/s eta 0:00:01
   ------------------------------------ --- 5.8/6.3 MB 9.3 MB/s eta 0:00:01
   ---------------

In [11]:
# Dependencies and Setup
import hvplot.pandas
import pandas as pd
import requests

# Import API key
from api_keys import geoapify_key

In [5]:
# Load the CSV file created in Part 1 into a Pandas DataFrame
city_data_df = pd.read_csv("output_data/cities.csv")

# Display sample data
city_data_df.head()

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
0,0,puerto baquerizo moreno,-0.9000,-89.6000,27.55,69,40,7.20,EC,1749490378
1,1,sitka,57.0531,-135.3300,8.94,81,100,1.54,US,1749490380
2,2,waitangi,-43.9535,-176.5597,10.70,65,96,13.71,NZ,1749490381
3,3,ushuaia,-54.8000,-68.3000,8.81,71,0,1.03,AR,1749490382
4,4,haiku-pauwela,20.9219,-156.3051,24.57,78,22,6.26,US,1749490383


---

### Step 1: Create a map that displays a point for every city in the `city_data_df` DataFrame. The size of the point should be the humidity in each city.

In [13]:
%%capture --no-display

# Configure the map plot
map_plot = city_data_df.hvplot.points(
    "Lng",
    "Lat",
    geo = True,
    size = "Humidity",
    scale = 1,
    color = "City",
    alpha = 0.5,
    tiles = "OSM",
    frame_width = 700,
    frame_height = 500
)

# Display the map
map_plot


:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Humidity)

### Step 2: Narrow down the `city_data_df` DataFrame to find your ideal weather condition

In [15]:
# Narrow down cities that fit criteria and drop any results with null values
narrowed_city_df = city_data_df.loc[
    (city_data_df["Max Temp"] < 27) & (city_data_df["Max Temp"] > 21) \
    & (city_data_df["Wind Speed"] < 4.5) \
    & (city_data_df["Cloudiness"] == 0)
]

# Drop any rows with null values
narrowed_city_df = narrowed_city_df.dropna()

# Display sample data
narrowed_city_df

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
69,69,pratt,37.6439,-98.7376,26.12,29,0,4.12,US,1749490460
112,112,cranbrook,49.4999,-115.7688,25.91,29,0,2.57,CA,1749490511
118,118,hauterive,46.0901,3.4459,22.97,47,0,2.16,FR,1749490518
165,165,tura,25.5198,90.2201,26.09,70,0,1.76,IN,1749490573
251,251,as salt,32.0392,35.7272,26.60,45,0,2.08,JO,1749490674
295,295,beira,-19.8436,34.8389,25.05,73,0,4.12,MZ,1749490726
316,316,carnarvon,-24.8667,113.6333,21.35,49,0,3.10,AU,1749490751
349,349,rexburg,43.8260,-111.7897,23.23,38,0,2.06,US,1749490790
357,357,tezu,27.9167,96.1667,24.27,87,0,2.14,IN,1749490801
383,383,arcachon,44.5875,-1.0519,26.03,47,0,3.60,FR,1749490834


### Step 3: Create a new DataFrame called `hotel_df`.

In [17]:
# Use the Pandas copy function to create DataFrame called hotel_df to store the city, country, coordinates, and humidity
hotel_df = narrowed_city_df[["City", "Country", "Lat", "Lng", "Humidity"]].copy()

# Add an empty column, "Hotel Name," to the DataFrame so you can store the hotel found using the Geoapify API
hotel_df["Hotel Name"] = ""

# Display sample data
hotel_df

,City,Country,Lat,Lng,Humidity,Hotel Name
69,pratt,US,37.6439,-98.7376,29,
112,cranbrook,CA,49.4999,-115.7688,29,
118,hauterive,FR,46.0901,3.4459,47,
165,tura,IN,25.5198,90.2201,70,
251,as salt,JO,32.0392,35.7272,45,
295,beira,MZ,-19.8436,34.8389,73,
316,carnarvon,AU,-24.8667,113.6333,49,
349,rexburg,US,43.8260,-111.7897,38,
357,tezu,IN,27.9167,96.1667,87,
383,arcachon,FR,44.5875,-1.0519,47,


### Step 4: For each city, use the Geoapify API to find the first hotel located within 10,000 metres of your coordinates.

In [19]:
# Set parameters to search for a hotel
radius = 1000
params = {
    "categories":"accommodation.hotel",
    "apiKey": geoapify_key,
    "limit":20
}

# Print a message to follow up the hotel search
print("Starting hotel search")

# Iterate through the hotel_df DataFrame
for index, row in hotel_df.iterrows():
    # get latitude, longitude from the DataFrame
    latitude = row["Lat"]
    longitude = row["Lng"]

    # Add the current city's latitude and longitude to the params dictionary
    params["filter"] = f"circle:{longitude},{latitude},{radius}"
    params["bias"] = f"proximity:{longitude},{latitude}"

    # Set base URL
    base_url = "https://api.geoapify.com/v2/places"

    # Make and API request using the params dictionary
    name_address = requests.get(base_url, params=params)

    # Convert the API response to JSON format
    name_address = name_address.json()

    # Grab the first hotel from the results and store the name in the hotel_df DataFrame
    try:
        hotel_df.loc[index, "Hotel Name"] = name_address["features"][0]["properties"]["name"]
    except (KeyError, IndexError):
        # If no hotel is found, set the hotel name as "No hotel found".
        hotel_df.loc[index, "Hotel Name"] = "No hotel found"

    # Log the search results
    print(f"{hotel_df.loc[index, 'City']} - nearest hotel: {hotel_df.loc[index, 'Hotel Name']}")

# Display sample data
hotel_df

Starting hotel search
pratt - nearest hotel: No hotel found
cranbrook - nearest hotel: No hotel found
hauterive - nearest hotel: No hotel found
tura - nearest hotel: No hotel found
as salt - nearest hotel: No hotel found
beira - nearest hotel: Hotel Savoy
carnarvon - nearest hotel: No hotel found
rexburg - nearest hotel: No hotel found
tezu - nearest hotel: No hotel found
arcachon - nearest hotel: No hotel found
ambovombe - nearest hotel: No hotel found
vernal - nearest hotel: Landmark Inn and Suites
archangelos - nearest hotel: Semina
saint-francois - nearest hotel: No hotel found
cabras - nearest hotel: Villa Canu
elmali - nearest hotel: Tu-Ba Hotel
chapaev - nearest hotel: No hotel found


,City,Country,Lat,Lng,Humidity,Hotel Name
69,pratt,US,37.6439,-98.7376,29,No hotel found
112,cranbrook,CA,49.4999,-115.7688,29,No hotel found
118,hauterive,FR,46.0901,3.4459,47,No hotel found
165,tura,IN,25.5198,90.2201,70,No hotel found
251,as salt,JO,32.0392,35.7272,45,No hotel found
295,beira,MZ,-19.8436,34.8389,73,Hotel Savoy
316,carnarvon,AU,-24.8667,113.6333,49,No hotel found
349,rexburg,US,43.8260,-111.7897,38,No hotel found
357,tezu,IN,27.9167,96.1667,87,No hotel found
383,arcachon,FR,44.5875,-1.0519,47,No hotel found


### Step 5: Add the hotel name and the country as additional information in the hover message for each city in the map.

In [21]:
%%capture --no-display

# Configure the map plot
map_plot = hotel_df.hvplot.points(
    "Lng",
    "Lat",
    geo = True,
    size = "Humidity",
    scale = 1,
    color = "City",
    alpha = 0.5,
    tiles = "OSM",
    frame_width = 700,
    frame_height = 500,
    hover_cols = ["Hotel Name", "Country"]
)

# Display the map
map_plot

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Humidity,Hotel Name,Country)